# RePro Walkthrough
**arXiv 2508.16671** — *RePro: Reflective Paper-to-Code Reproduction Enabled by Fine-Grained Verification*

This notebook traces paper sections to code and runs offline sanity checks (no LLM calls, no API keys needed).

## §3.1 — Supervisory Signal Design

The paper extracts a **fingerprint**: a set of atomic `<fact>…</fact> <scope>…</scope>` criteria.

Three steps:
1. **Guide Extraction** — 3 levels (framework, configuration, paragraph-by-paragraph sentence selection)
2. **Source Grounding** — top-3 paragraphs per criterion via embedding retrieval
3. **Standardization → Filtering** — fact-scope decomposition, cluster dedup + LLM semantic filter

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from repro.supervisory_signal import (
    Criterion, _split_sentences, _index_paragraph, _parse_criteria
)

# Criterion data structure
c = Criterion(fact='learning rate is 0.001', scope='optimizer configuration')
assert str(c) == '<fact>learning rate is 0.001</fact> <scope>optimizer configuration</scope>'
print('Criterion format:', c)

In [ ]:
# Sentence indexing (used in Guide Extraction prompt — Figure 6)
para = 'The model uses GCN layers. This is effective. We use AdamW with lr=0.01.'
indexed = _index_paragraph(para)
print(indexed)
assert '[1]:' in indexed and '[3]:' in indexed

In [ ]:
# Criteria parsing from LLM output
raw = '<fact>learning rate is 0.01</fact> <scope>AdamW optimizer</scope>\n<fact>batch size is 64</fact> <scope>training loop</scope>'
criteria = _parse_criteria(raw)
assert len(criteria) == 2
assert criteria[0].fact == 'learning rate is 0.01'
assert criteria[1].scope == 'training loop'
print('Parsed', len(criteria), 'criteria')

## §3.2 — Reflective Code Development

Loop: **verify** (per criterion, pass/fail) → **plan** (CONFIG PLAN + CODE PLAN) → **refine** (targeted edits).

Max 4 iterations (§4.1). Early-stop when all criteria pass.

In [ ]:
from repro.code_development import (
    VerificationResult, IterationRecord, RevisionPlan, ReProPipeline,
    _parse_revision_plan, _parse_code_blocks, _summarize_code
)

# Verify max iterations constant
assert ReProPipeline.MAX_ITERATIONS == 4, 'Must be 4 per §4.1'
print('MAX_ITERATIONS:', ReProPipeline.MAX_ITERATIONS)

In [ ]:
# IterationRecord.all_passed logic
c1 = Criterion(fact='lr=0.01', scope='optimizer')
c2 = Criterion(fact='batch=64', scope='dataloader')

results_all_pass = [
    VerificationResult(criterion=c1, status='PASS', feedback='correct'),
    VerificationResult(criterion=c2, status='PASS', feedback='correct'),
]
results_one_fail = [
    VerificationResult(criterion=c1, status='PASS', feedback='correct'),
    VerificationResult(criterion=c2, status='FAIL', feedback='batch size is 32, not 64'),
]

rec_pass = IterationRecord(iteration=1, results=results_all_pass, plan=None)
rec_fail = IterationRecord(iteration=1, results=results_one_fail, plan=None)

assert rec_pass.all_passed
assert not rec_fail.all_passed
assert rec_fail.n_fail == 1
print('all_passed logic: OK')

In [ ]:
# Revision plan parsing (CONFIG PLAN + CODE PLAN sections)
raw_plan = '''## CONFIG PLAN
- File: configs/base.yaml
- Change: set learning_rate to 0.01

## CODE PLAN
- File: model.py
- Function: train_step
- Change: use AdamW instead of Adam'''

config_plan, code_plan = _parse_revision_plan(raw_plan)
assert 'learning_rate' in config_plan
assert 'AdamW' in code_plan
print('CONFIG PLAN:', config_plan[:40])
print('CODE PLAN:', code_plan[:40])

In [ ]:
# Code block parsing (LLM output format)
llm_output = '''```python:model.py
class MyModel:\n    pass\n```\n```python:train.py\ndef train():\n    pass\n```'''
files = _parse_code_blocks(llm_output)
assert 'model.py' in files
assert 'train.py' in files
print('Parsed files:', list(files.keys()))

## §3.1 — Source Grounding (embedding-based, top-3)

In [ ]:
import numpy as np
from repro.embeddings import embed, top_k_indices, cosine_similarity_matrix

# Verify normalisation
vecs = embed(['hello world', 'learning rate is 0.01', 'batch size 64'])
norms = np.linalg.norm(vecs, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5), 'Embeddings must be L2-normalised'
print('Embedding shape:', vecs.shape, '  Norms:', norms.round(4))

In [ ]:
# top_k_indices returns top-3 (§3.1)
paragraphs = [
    'The optimizer used is AdamW with learning rate 0.001.',
    'The weather was sunny that day.',
    'We use a batch size of 64 during training.',
    'Future work may explore other architectures.',
    'The model is trained for 100 epochs on CIFAR-10.',
]
corpus = embed(paragraphs)
query = embed(['learning rate and optimizer settings'])
idxs = top_k_indices(query[0], corpus, k=3)
print('Top-3 paragraphs for query:', idxs)
for i in idxs:
    print(f'  [{i}]', paragraphs[i])

## Prompts (Appendix Figs 6-16)

All LLM prompts are in `repro/prompts.py` as module-level string constants.

In [ ]:
from repro import prompts

# Verify all expected prompts are present
expected = [
    'GUIDE_EXTRACTION_SYSTEM',    # Fig 6
    'STANDARDIZATION_SYSTEM',     # Figs 7-8
    'FRAMEWORK_GUIDE_SYSTEM',     # Fig 9
    'CONFIGURATION_GUIDE_SYSTEM', # Fig 10
    'SEMANTIC_FILTER_SYSTEM',     # Fig 11
    'SKELETON_SYSTEM',            # Fig 12
    'FILL_SYSTEM',                # Fig 13
    'VERIFICATION_SYSTEM',        # Fig 14
    'REVISION_PLANNING_SYSTEM',   # Figs 15-16
    'REFINEMENT_SYSTEM',          # §3.2
]
for name in expected:
    assert hasattr(prompts, name), f'Missing prompt: {name}'
print('All', len(expected), 'prompt constants present')

## Summary

All offline checks pass. To run the full RePro pipeline, supply LLM API keys and call `run_repro()` from `repro/pipeline.py`. See `README.md` for details.